In [1]:
# Uwaga! 
# Notatnik każdorazowo usuwa kopiuje dane do tabeli i usuwa pliki zapisane przez Invoke SSIS Activity


# KONFIGURACJA
LOGGING_PATH = "/lakehouse/default/Files/Logging"  # ścieżka lokalna 
FILE_PREFIX  = "event_messages_"                   # prefix pliku CSV
TARGET_TABLE = "ssis_event_messages"               # nazwa tabeli Delta

StatementMeta(, e1f2b5e9-31f8-4f38-bf6a-ff3f29c475ed, 3, Finished, Available, Finished, False)

In [2]:
import os
from pyspark.sql import functions as F

StatementMeta(, e1f2b5e9-31f8-4f38-bf6a-ff3f29c475ed, 4, Finished, Available, Finished, False)

In [3]:
csv_paths = []

for folder_name in os.listdir(LOGGING_PATH):
    folder_path = os.path.join(LOGGING_PATH, folder_name)
    if not os.path.isdir(folder_path):
        continue
    for file_name in os.listdir(folder_path):
        if file_name.startswith(FILE_PREFIX) and file_name.endswith(".csv"):
            spark_path = "file://" + os.path.join(folder_path, file_name)
            csv_paths.append(spark_path)
            print(f"[OK] {folder_name}/{file_name}")

if not csv_paths:
    raise FileNotFoundError(f"Brak plików '{FILE_PREFIX}*.csv' w {LOGGING_PATH}")



df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")   
    .option("escape", '"')
    .csv(csv_paths)
)

print(f"Wczytano łącznie {df_raw.count()} wierszy.")
df_raw.printSchema()


StatementMeta(, e1f2b5e9-31f8-4f38-bf6a-ff3f29c475ed, 5, Finished, Available, Finished, False)

FileNotFoundError: Brak plików 'event_messages_*.csv' w /lakehouse/default/Files/Logging

In [ ]:
# ============================================================
# Czyszczenie i zapis do tabeli Delta
# ============================================================

df_clean = (
    df_raw
    .withColumn("MessageTime", F.to_timestamp("MessageTime"))
    .withColumn("_load_timestamp", F.current_timestamp())
)

df_clean.show(3, truncate=100)


(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

count = spark.table(TARGET_TABLE).count()
print(f"Dane zapisane do tabeli '{TARGET_TABLE}'")
print(f"Liczba wierszy w tabeli: {count}")

StatementMeta(, e1f2b5e9-31f8-4f38-bf6a-ff3f29c475ed, -1, Cancelled, , Cancelled, True)

In [ ]:
import os
import shutil


folders = [
    f for f in os.listdir(LOGGING_PATH)
    if os.path.isdir(os.path.join(LOGGING_PATH, f))
]

print(f"Znaleziono {len(folders)} folder(ów) do usunięcia:")

deleted = 0
errors  = 0

for folder_name in folders:
    folder_path = os.path.join(LOGGING_PATH, folder_name)
    try:
        shutil.rmtree(folder_path)
        print(f"  [OK] Usunięto: {folder_name}")
        deleted += 1
    except Exception as e:
        print(f"  [ERROR] {folder_name}: {e}")
        errors += 1


    print(f"\nUsunięto: {deleted} | Błędy: {errors}")


StatementMeta(, e1f2b5e9-31f8-4f38-bf6a-ff3f29c475ed, -1, Cancelled, , Cancelled, True)